# Linking Energy and the Economy: MESSAGE-MACRO

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 06 · Mini integrated assessment model series, notebook 4 of 5 · Requires coding exercises 04 and 05

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/06-message-macro.ipynb)

In notebooks 04 and 05, demand for energy services was fixed: however high the carbon price, the world consumed the same amount of light, heat, and transport. In reality, higher energy prices lead households and firms to use less energy, and spending more on energy leaves less for consumption and investment elsewhere. Both effects matter for the cost of climate policy.

MESSAGEix captures them by linking the energy model **MESSAGE** to a macro-economic growth model, **MACRO**. The two models are solved in turn, passing prices and costs in one direction and demands in the other, until they agree. This notebook builds the linkage exactly as MESSAGEix does, first on Westeros, where every step can be followed by hand, and then on the global model of notebook 05.

| Notebook | Content |
|---|---|
| 03 | Linear optimisation, shadow prices, and a small capacity expansion model |
| 04 | Westeros: the MESSAGE formulation on a small example |
| 05 | World: the same formulation applied to a global energy system |
| **06 (this notebook)** | **MESSAGE-MACRO: linking the energy model to a macro-economic growth model** |
| 07 | From the mini model to the real `message_ix` package |

Prompts: **💬 Predict**, **✏️ Exercise** (replace every `...` or `___`; self-checks print ✅ or ❌), and **💬 Question**.

## Learning objectives

On completing this notebook, students should be able to:

1. Describe the structure of MACRO: a single-sector growth model with a nested production function of capital, labour, and energy services.
2. Explain the information exchanged between MESSAGE and MACRO, and why the two models are solved iteratively.
3. Explain what calibration of MACRO to a baseline means, and verify it.
4. Quantify the GDP loss and the demand reduction caused by a climate policy, and explain how the elasticity of substitution affects both.
5. State the main limitations of a MESSAGE-MACRO estimate of the cost of climate policy.

## Structure of the notebook

| Part | Content | Approximate time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | What MACRO is | 15 min |
| 2 | Calibrating MACRO to the Westeros baseline | 15 min |
| 3 | The MESSAGE-MACRO iteration on Westeros | 20 min |
| 4 | Inside the iteration (★ Stretch) | 20 min |
| 5 | MESSAGE-MACRO for the world | 25 min |
| 6 | What MESSAGE-MACRO does and does not measure | 10 min |

---
# Part 0 · Setup

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyomo highspy
    !wget --quiet --timestamping https://raw.githubusercontent.com/LUMS-WIT/climate-futures/main/content-src/coding/miniiam.py

In [ ]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import miniiam as mi
from miniiam import make_df

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.precision", 3)

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("✅ " if passed else "❌ ") + description)

---
# Part 1 · What MACRO is

MACRO describes the whole economy of a region as a single **representative producer–consumer** who decides, in every period, how much of the economy's output to consume and how much to invest, so as to maximise the discounted sum of the logarithm of consumption (its *utility*). This is the Ramsey growth model, the same framework that underlies DICE (Lecture 9).

Output is produced from **capital** $K$, **labour** $L$, and **energy services** $E_s$ (one for each demand sector of MESSAGE) with a nested constant-elasticity-of-substitution (CES) production function:

$$
Y = \Big( a \, \big(K^{\alpha} L^{1-\alpha}\big)^{\rho} + \sum_s b_s \, E_s^{\rho} \Big)^{1/\rho},
\qquad \rho = \frac{\sigma - 1}{\sigma}
$$

where $\sigma$ (`esub`) is the **elasticity of substitution** between the capital–labour bundle and energy. With a small $\sigma$, energy is hard to replace: cutting energy use reduces output sharply. With a larger $\sigma$, capital (insulation, efficient machines) can substitute for energy more easily.

Every period, output is used for three purposes:

$$
Y = C + I + EC
$$

consumption $C$, investment in capital $I$, and the cost of the energy system $EC$, which MACRO takes from MESSAGE. This is the link: if climate policy makes the energy system more expensive, less is left for consumption and investment, and the economy grows more slowly.

| MACRO variable (as in `macro_core.gms`) | Meaning |
|---|---|
| `K`, `KN` | Capital stock; new capital |
| `Y`, `YN` | Production; new production |
| `PHYSENE` | Physical use of energy services (the demand passed back to MESSAGE) |
| `PRODENE`, `NEWENE` | Energy services in the production function; new energy services |
| `C`, `I` | Consumption; investment |
| `EC` | Energy-system cost, approximated around the MESSAGE solution |
| `UTILITY` | Discounted sum of the logarithm of consumption |

The equations are implemented in `mi.solve_macro`. The cell below prints the part that evaluates them, so that each line can be matched to an equation of the [MESSAGEix MACRO documentation](https://docs.messageix.org/en/latest/model/MACRO/macro_core.html).

In [ ]:
src = inspect.getsource(mi.solve_macro)
print(src[src.index("    def unpack"):src.index("    def utility")])

**💬 Question.** The energy cost `EC` in MACRO is not simply the MESSAGE system cost. It is a quadratic approximation around the MESSAGE solution, using the MESSAGE demand prices. Why does MACRO need to know how the energy cost would change if demand changed, rather than only the cost at the current demand?

*Answer:* *(write the answer here)*

---
# Part 2 · Calibrating MACRO to the Westeros baseline

Before the linked model can be used, MACRO must be **calibrated** so that, with baseline energy prices and costs, it reproduces the baseline: the GDP path assumed for the scenario and the energy-service demands used in MESSAGE. MESSAGEix does this with `Scenario.add_macro()`, which:

1. derives the production-function coefficients $a$ and $b_s$ from base-year GDP, demand, and prices;
2. adjusts the potential GDP growth rates (`grow`) and the autonomous energy efficiency improvement rates (`aeei`) in alternating iterations until MACRO reproduces the baseline;
3. checks that the calibrated MESSAGE-MACRO model converges immediately on the baseline.

The input data have the same names as in `message_ix`. For Westeros they are those of the MESSAGEix test suite.

In [ ]:
base = mi.build_westeros(mi.Scenario(model="Westeros Electrified", scenario="baseline"))
base.solve()

macro_data = mi.westeros_macro_data()
for name in ["gdp_calibrate", "price_ref", "demand_ref", "esub", "kpvs", "kgdp", "depr", "drate"]:
    print(name, "\n", macro_data[name].drop(columns=[c for c in ["node", "unit"] if c in macro_data[name]]).to_string(index=False), "\n")

| Parameter | Meaning |
|---|---|
| `gdp_calibrate` | GDP path of the baseline, in **billion** USD (the unit MESSAGE-MACRO expects); values before the first model year set the base year |
| `price_ref`, `demand_ref`, `cost_ref` | Price (USD/kWa) and level (GWa) of each energy service, and total energy-system cost (billion USD), in the base year |
| `esub` | Elasticity of substitution between capital–labour and energy |
| `kpvs` | Capital value share (the exponent α) |
| `kgdp` | Ratio of capital stock to GDP in the base year |
| `depr`, `drate` | Depreciation rate of capital; discount rate |
| `aeei` | First guess of the autonomous energy efficiency improvement, adjusted by calibration |

In [ ]:
macro = base.add_macro(macro_data)

### ✏️ Exercise 1: Is the calibration successful?

After calibration, `macro.var("GDP")` holds MACRO's GDP and `macro.var("PHYSENE")` its demand for light. Compare them with `gdp_calibrate` and with the MESSAGE demand of the baseline, and store the largest relative deviation of each in `gdp_dev` and `demand_dev`.

In [ ]:
gdp_target = macro_data["gdp_calibrate"].set_index("year")["value"]
gdp_model = macro.var("GDP").set_index("year")["lvl"]
gdp_dev = (gdp_model / gdp_target.loc[gdp_model.index] - 1).abs().max()

demand_target = base.par("demand").set_index("year")["value"] / 1000     # GWa -> TWa
demand_model = macro.var("PHYSENE").set_index("year")["lvl"]
demand_dev = ___
print(f"largest deviation: GDP {gdp_dev:.1e}, demand {demand_dev:.1e}")

In [ ]:
check("MACRO reproduces baseline GDP within 0.1%", lambda: gdp_dev < 1e-3)
check("MACRO reproduces baseline demand within 0.1%", lambda: demand_dev < 1e-3)

In [ ]:
calibrated = pd.DataFrame({"grow (potential GDP growth, %/yr)": {y: 100 * macro.macro["grow"][y] for y in macro.macro["years"]},
                           "aeei (energy efficiency improvement, %/yr)": {y: 100 * macro.macro["aeei"]["light", y] for y in macro.macro["years"]}})
calibrated.round(2)

**💬 Question.** The calibrated `aeei` values are not the 2% per year given as input. What do they represent, and why does the calibration change them?

*Answer:* *(write the answer here)*

---
# Part 3 · The MESSAGE-MACRO iteration on Westeros

The emission bound of notebook 04 (500 Mt CO₂ on average) is now applied to the calibrated scenario and solved with `model="MESSAGE-MACRO"`. Each iteration:

1. solves MESSAGE with the current demands;
2. passes the demand levels, the demand prices (`PRICE_COMMODITY`), and the total system cost (`COST_NODAL_NET`) to MACRO;
3. solves MACRO, which chooses a new level of energy service;
4. rescales the MESSAGE demands accordingly, by at most ±20% per iteration (`max_adjustment`).

The iteration stops when the largest relative change in demand is below 1% (`convergence_criterion`).

In [ ]:
def add_bound(s, name, value=500.0):
    out = s.clone(scenario=name)
    out.add_par("bound_emission", make_df("bound_emission", node="Westeros", type_emission="GHG", type_tec="all",
                                          type_year="cumulative", value=value, unit="MtCO2"))
    return out


bound_msg = add_bound(base, "emission_bound (MESSAGE)")
bound_msg.solve()                                   # demand fixed

bound_mm = add_bound(macro, "emission_bound (MESSAGE-MACRO)")
bound_mm.solve(model="MESSAGE-MACRO")               # demand responds to prices
bound_mm.iterations

**💬 Predict before running the next cell.** Compared with the MESSAGE-only run, will the MESSAGE-MACRO run have a higher or lower demand for light, and a higher or lower carbon price?

*Answer:* *(write the answer here)*

In [ ]:
compare = pd.DataFrame({
    "demand, MESSAGE (GWa)": bound_msg.var("DEMAND").set_index("year")["lvl"],
    "demand, MESSAGE-MACRO (GWa)": bound_mm.var("DEMAND").set_index("year")["lvl"],
    "carbon price, MESSAGE (USD/t)": bound_msg.var("PRICE_EMISSION").set_index("year")["lvl"],
    "carbon price, MESSAGE-MACRO (USD/t)": bound_mm.var("PRICE_EMISSION").set_index("year")["lvl"],
    "GDP loss (%)": 100 * (1 - bound_mm.var("GDP").set_index("year")["lvl"] / macro.var("GDP").set_index("year")["lvl"]),
})
compare.round(2)

As in MESSAGEix, `GDP` is reported in billion USD, while the MACRO variables `C`, `I`, `K`, and `EC` are in trillion USD. Westeros has a GDP of about 1,000 billion USD and an electricity system that costs about 10 billion USD per year, so a policy that raises the cost of electricity has a small but visible effect on GDP. The demand response is larger, because it depends on the change in the *price* of light rather than on the share of electricity in GDP.

### ✏️ Exercise 2: How elastic is the economy?

The elasticity of substitution `esub` is one of the most uncertain MACRO parameters. Recalibrate MACRO with `esub = 0.5` instead of 0.2 (copy `macro_data` and replace the value), apply the same emission bound, and solve with MESSAGE-MACRO. Store the scenario as `bound_hi`, and compute the reduction of demand in 720 relative to the baseline for both elasticities (`cut_lo` for 0.2, `cut_hi` for 0.5, in %).

In [ ]:
data_hi = dict(macro_data)
data_hi["esub"] = pd.DataFrame({"node": ["Westeros"], "value": [___]})
macro_hi = base.add_macro(data_hi)
bound_hi = add_bound(macro_hi, "emission_bound esub=0.5")
bound_hi.solve(model=___)

d0 = base.var("DEMAND").set_index("year")["lvl"]
cut_lo = 100 * (1 - bound_mm.var("DEMAND").set_index("year")["lvl"][720] / d0[720])
cut_hi = ___
print(f"demand reduction in 720: {cut_lo:.1f}% (esub 0.2), {cut_hi:.1f}% (esub 0.5)")

In [ ]:
check("a higher elasticity gives a larger demand response", lambda: cut_hi > cut_lo)

With `esub = 0.5`, the iteration log shows the demand adjustment changing sign from one iteration to the next. This is the **oscillation** described in the MESSAGEix documentation: MESSAGE and MACRO alternate between two solutions. The algorithm detects it and divides the permitted adjustment step by √2 each time (the `scaling` column of `bound_hi.iterations`) until the models agree.

**💬 Question.** A higher elasticity of substitution leads to a larger reduction of energy demand. Does it lead to a larger or a smaller GDP loss? Explain the economic intuition, and say why the choice of this parameter is politically significant.

*Answer:* *(write the answer here)*

---
# Part 4 · Inside the iteration (★ Stretch)

`solve(model="MESSAGE-MACRO")` hides the iteration. This part runs it by hand, using the same three functions that `solve` uses, so that each step of `MESSAGE-MACRO_run.gms` becomes visible.

### ✏️ Exercise 3: Iterate by hand · ★ Stretch

Complete the loop below. In each iteration: solve MESSAGE (`mi.solve_message`), extract the information for MACRO (`mi.message_to_macro`), solve MACRO (`mi.solve_macro`), compute the demand scale factor `PHYSENE / enestart` (limited to 0.8–1.2), and multiply the MESSAGE demand by it. Print the largest adjustment in each iteration.

In [ ]:
manual = add_bound(macro, "by hand")
mac = manual.macro
for it in range(1, 11):
    manual.solution = mi.solve_message(manual)
    link = mi.message_to_macro(manual, mac)
    result = mi.solve_macro(mac, link)
    scale = {k: min(max(result[___][k] / link[___][k], 0.8), 1.2) for k in link["enestart"]}
    largest = max(abs(v - 1) for v in scale.values())
    print(f"iteration {it}: largest demand adjustment {largest:.4f}")
    if largest < 0.01:
        break
    dem = manual.pars["demand"]
    for (sector, year), factor in scale.items():
        mask = dem["year"] == year
        dem.loc[mask, "value"] = dem.loc[mask, "value"] * ___
print(manual.solution["DEMAND"][["year", "lvl"]])

In [ ]:
check("the manual iteration reaches the same demand as solve()",
      lambda: np.allclose(manual.solution["DEMAND"]["lvl"].values, bound_mm.var("DEMAND")["lvl"].values, rtol=0.02))

---
# Part 5 · MESSAGE-MACRO for the world

The same steps apply to the global model of notebook 05, with three MACRO sectors (`spec`, `therm`, `transport`) mapped to the three useful-energy demands. Base-year GDP is about 85 trillion USD (2020, market exchange rates), and the baseline GDP grows along an SSP2-like path. Calibration takes a few seconds.

In [ ]:
world = mi.build_world(mi.Scenario(model="World", scenario="baseline"))
world.solve()
world_data = mi.world_macro_data()
(world_data["gdp_calibrate"].set_index("year")["value"] / 1000).to_frame("GDP (trillion USD)").T

In [ ]:
world_macro = world.add_macro(world_data)

In [ ]:
horizon_years = 80

def budget(s, gt, name):
    out = s.clone(scenario=name)
    out.add_par("bound_emission", make_df("bound_emission", node="World", type_emission="GHG", type_tec="all",
                                          type_year="cumulative", value=gt * 1000 / horizon_years, unit="MtCO2"))
    return out

w2c_msg = budget(world, 1150, "2C, MESSAGE")
w2c_msg.solve()
w2c_mm = budget(world_macro, 1150, "2C, MESSAGE-MACRO")
w2c_mm.solve(model="MESSAGE-MACRO")

### ✏️ Exercise 4: The macro-economic cost of the 2 °C budget

Compute, for every model year, the GDP loss and the consumption loss of `w2c_mm` relative to `world_macro`, in percent, and store them in a DataFrame `losses` with columns `"GDP loss (%)"` and `"consumption loss (%)"`. Then compute the change in each demand sector relative to the baseline (in %) as `demand_change`, a table with years as rows and sectors as columns.

In [ ]:
gdp0 = world_macro.var("GDP").set_index("year")["lvl"]
gdp1 = w2c_mm.var("GDP").set_index("year")["lvl"]
c0 = world_macro.var("C").set_index("year")["lvl"]
c1 = w2c_mm.var(___).set_index("year")["lvl"]
losses = pd.DataFrame({"GDP loss (%)": 100 * (1 - gdp1 / gdp0),
                       "consumption loss (%)": ___})

d0 = world.var("DEMAND").pivot(index="year", columns="commodity", values="lvl")
d1 = w2c_mm.var("DEMAND").pivot(index="year", columns="commodity", values="lvl")
demand_change = ___
pd.concat([losses, demand_change], axis=1).round(2)

In [ ]:
check("the 2 °C budget reduces GDP in 2100", lambda: losses.loc[2100, "GDP loss (%)"] > 0)
check("all demands fall relative to the baseline in 2050", lambda: (demand_change.loc[2050] < 0).all())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for s, style in [(w2c_msg, "--"), (w2c_mm, "-")]:
    p = s.var("PRICE_EMISSION").set_index("year")["lvl"]
    p.plot(ax=axes[0], style=style, marker="o", label=s.scenario)
    (s.var("EMISS").set_index("year")["lvl"] / 1000).plot(ax=axes[1], style=style, marker="o", label=s.scenario)
axes[0].set(title="Carbon price (USD per t CO2)")
axes[1].set(title="CO2 emissions (Gt per year)")
axes[1].axhline(0, color="grey", lw=0.5)
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**💬 Question.** In the World model, which demand sector responds most strongly to the 2 °C budget, and why? Relate your answer to the change in the price of each energy service between the baseline and the policy scenario.

*Answer:* *(write the answer here)*

---
# Part 6 · What MESSAGE-MACRO does and does not measure

MESSAGE-MACRO adds an important feedback to the energy model: demand responds to prices, and energy costs are reflected in GDP and consumption. Its results are widely used, for example in the IPCC's estimates of the macro-economic cost of mitigation. The estimates rest on strong simplifications, however:

- **No climate damages.** The model compares a policy scenario with a baseline in which climate change has no economic cost. The GDP loss is the cost of mitigation, not the net effect of climate policy.
- **A single good and a single consumer.** There is no distinction between sectors (other than energy services), no trade, no unemployment, and no distribution of costs between rich and poor.
- **Perfect foresight and optimal savings.** The representative agent saves and invests optimally over the whole century.
- **Calibrated, not estimated.** The parameters (`esub`, `kpvs`, `kgdp`) are chosen to reproduce a baseline, not estimated from data, and the results are sensitive to them (Part 3).

**💬 Final question.** A newspaper reports that "a model shows that limiting warming to 2 °C would cost less than 1% of global GDP". Using this notebook, write three sentences that a careful reader should add to that headline.

*Answer:* *(write the answer here)*

---
# References and further reading

- Messner, S., and Schrattenholzer, L. (2000). MESSAGE–MACRO: linking an energy supply model with a macroeconomic module and solving it iteratively. *Energy*, 25(3), 267–282. [doi:10.1016/S0360-5442(99)00063-8](https://doi.org/10.1016/S0360-5442(99)00063-8)
- Manne, A. S., and Richels, R. G. (1992). *Buying Greenhouse Insurance: The Economic Costs of CO₂ Emission Limits.* MIT Press, Cambridge, MA.
- [MESSAGEix documentation: MACRO](https://docs.messageix.org/en/latest/model/MACRO/macro_core.html) and [Calibrate and tune MESSAGE-MACRO](https://docs.messageix.org/en/latest/macro.html).
- The MESSAGEix source code: `message_ix/model/MACRO/macro_core.gms`, `macro_calibration.gms`, and `MESSAGE-MACRO_run.gms` ([GitHub](https://github.com/iiasa/message_ix/tree/main/message_ix/model)).